# 02 — Feature Engineering & Frequency Alignment
Align daily + monthly data, build the feature matrix for all models.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120

## 1. Load raw data

In [ ]:
prices = pd.read_csv('../data/raw/daily_prices.csv', index_col=0, parse_dates=True)
macro  = pd.read_csv('../data/raw/monthly_macro.csv', index_col=0, parse_dates=True)

prices.index = prices.index.tz_localize(None)  # strip tz if present
print('Daily shape:', prices.shape)
print('Monthly shape:', macro.shape)

## 2. Daily features

In [ ]:
df = pd.DataFrame(index=prices.index)

# Target
df['silver']         = prices['silver']
df['silver_return']  = np.log(prices['silver']).diff()   # log-return

# Price-based covariates
df['gold_return']    = np.log(prices['gold']).diff()
df['copper_return']  = np.log(prices['copper']).diff()
df['usd_return']     = np.log(prices['usd_index']).diff()
df['sp500_return']   = np.log(prices['sp500']).diff()
df['gs_ratio']       = prices['gold'] / prices['silver']  # gold/silver ratio

# Lags of silver return (AR terms for ARIMAX)
for lag in [1, 2, 3, 5, 10]:
    df[f'silver_lag{lag}'] = df['silver_return'].shift(lag)

# Rolling volatility (realised vol proxy)
df['silver_vol_5d']  = df['silver_return'].rolling(5).std()
df['silver_vol_21d'] = df['silver_return'].rolling(21).std()

# Rolling momentum
df['mom_5d']         = df['silver_return'].rolling(5).sum()
df['mom_21d']        = df['silver_return'].rolling(21).sum()

print(df.shape)
df.head()

## 3. Align monthly macro to daily (forward-fill)
Forward-fill is the correct approach: a monthly CPI reading published on day T is known for all days until the next release.

In [ ]:
# Reindex monthly macro to daily, then forward-fill
macro_daily = macro.reindex(df.index, method='ffill')

# Add month-over-month changes (stationary version of levels)
for col in macro.columns:
    macro_daily[f'{col}_chg'] = macro[col].pct_change().reindex(df.index, method='ffill')

df = pd.concat([df, macro_daily], axis=1)
print('Combined shape:', df.shape)

## 4. Add sentiment (placeholder — fill in after 06_sentiment.ipynb)

In [ ]:
import os
sentiment_path = '../data/processed/daily_sentiment.csv'
if os.path.exists(sentiment_path):
    sentiment = pd.read_csv(sentiment_path, index_col=0, parse_dates=True)
    df = df.join(sentiment, how='left')
    print('Sentiment joined:', sentiment.columns.tolist())
else:
    print('No sentiment file yet — run 06_sentiment.ipynb first.')
    df['sentiment_score'] = np.nan

## 5. Train / validation / test split
- Train: 2015–2021
- Val:   2022
- Test:  2023–2024


In [ ]:
df = df.dropna(subset=['silver_return'])

train = df[df.index < '2022-01-01']
val   = df[(df.index >= '2022-01-01') & (df.index < '2023-01-01')]
test  = df[df.index >= '2023-01-01']

print(f'Train: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} days)')
print(f'Val:   {val.index[0].date()} → {val.index[-1].date()}  ({len(val)} days)')
print(f'Test:  {test.index[0].date()} → {test.index[-1].date()}  ({len(test)} days)')

## 6. Save feature matrix

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

df.to_csv('../data/processed/features.csv')
train.to_csv('../data/processed/train.csv')
val.to_csv('../data/processed/val.csv')
test.to_csv('../data/processed/test.csv')

print('Saved feature matrix and splits to data/processed/')